# Part 7: Hyperparameter Optimization
## Optuna-based HPO for GNN Models

Uses Optuna to find optimal hyperparameters for GNN models:
- Hidden dimension: [64, 128, 256]
- Number of layers: [2, 3, 4, 5]
- Learning rate: [1e-5, 1e-2] (log scale)
- Dropout: [0.1, 0.5]
- Attention heads: [4, 8] (for GAT-based models)

In [ ]:
# @title 1. Setup
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("No GPU")

train_df = pd.read_csv('data/train.csv')
val_df = pd.read_csv('data/val.csv')

In [ ]:
# @title 2. Create DataLoaders
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from vegfr2.features import mol_to_graph_with_fps

def make_loader(df, batch_size=128, shuffle=False):
    data_list = []
    for s, y in zip(df['smiles'], df['active'].astype(int)):
        try:
            g = mol_to_graph_with_fps(s, use_morgan=True, use_maccs=True)
            data = Data(x=g['node_feats'], edge_index=g['edge_index'],
                       edge_attr=g['edge_feats'],
                       y=torch.tensor([y], dtype=torch.float32))
            data_list.append(data)
        except:
            pass
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(train_df, shuffle=True)
val_loader = make_loader(val_df)

In [ ]:
# @title 3. Optuna HPO
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    print("Installing optuna...")
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

from vegfr2.gnn_pyg import build_pyg_model
from vegfr2.metrics import classification_metrics

def objective(trial, model_name):
    hidden = trial.suggest_categorical('hidden', [64, 128, 256])
    layers = trial.suggest_int('layers', 2, 5)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    heads = trial.suggest_categorical('heads', [4, 8]) if model_name in ['gat', 'gatv2', 'graph_transformer'] else 8
    
    model = build_pyg_model(model_name, in_dim=2246, hidden=hidden, layers=layers, heads=heads, dropout=dropout).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    
    for _ in range(50):
        model.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = loss_fn(logits.squeeze(), batch.y)
            opt.zero_grad()
            loss.backward()
            opt.step()
    
    model.eval()
    val_probs, val_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            logits = model(batch.x, batch.edge_index, batch.batch)
            val_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
            val_true.extend(batch.y.squeeze().cpu().numpy().astype(int))
    
    return classification_metrics(val_true, val_probs).get('auc') or 0.0

models_to_hpo = ['gin', 'pna', 'graph_transformer']
best_params = {}

for model_name in models_to_hpo:
    print(f"\nHPO for {model_name.upper()}...")
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, model_name), n_trials=30)
    
    best_params[model_name] = study.best_params
    print(f"  Best AUC: {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")

In [ ]:
# @title 4. Results Summary
print("\n" + "=" * 60)
print("HPO RESULTS")
print("=" * 60)

for model_name, params in best_params.items():
    print(f"\n{model_name.upper()}:")
    for k, v in params.items():
        print(f"  {k}: {v}")

In [ ]:
# @title 5. Visualization
fig, axes = plt.subplots(1, len(models_to_hpo), figsize=(5*len(models_to_hpo), 4))

for idx, model_name in enumerate(models_to_hpo):
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, model_name), n_trials=30)
    
    trials = study.trials
    ax = axes[idx] if len(models_to_hpo) > 1 else axes
    ax.plot([t.value for t in trials], 'o-', alpha=0.5)
    ax.axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
    ax.set_title(f'{model_name.upper()} HPO')
    ax.set_xlabel('Trial')
    ax.set_ylabel('AUC')
    ax.legend()

plt.tight_layout()
plt.savefig('images/hpo_results.png', dpi=150, bbox_inches='tight')
plt.show()